# 04 — Inference Pipeline

End-to-end prediction pipeline using the trained **LightGBM + TF-IDF** models.

Given a list of names, this notebook:
1. Loads the saved LightGBM models and TF-IDF vectorizers
2. Demonstrates single-name and batch prediction
3. Provides a reusable `predict_demographics()` function
4. Shows confidence-threshold filtering for downstream use

**Models loaded from `models/`:**
- `gender_lgbm_model.pkl` + `gender_tfidf_vectorizer.pkl`
- `race_lgbm_model.pkl` + `race_tfidf_vectorizer.pkl`
- `race_label_encoder.pkl`

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# ── Paths ──────────────────────────────────────────────────────────────────────
BASE       = os.path.abspath(os.path.join(os.getcwd(), '..'))
MODELS_DIR = os.path.join(BASE, 'models')
print('Base:', BASE)

## 1. Load Models

In [ ]:
# ── LightGBM + TF-IDF ─────────────────────────────────────────────────────────
gender_vec   = joblib.load(os.path.join(MODELS_DIR, 'gender_tfidf_vectorizer.pkl'))
gender_model = joblib.load(os.path.join(MODELS_DIR, 'gender_lgbm_model.pkl'))

race_vec     = joblib.load(os.path.join(MODELS_DIR, 'race_tfidf_vectorizer.pkl'))
race_model   = joblib.load(os.path.join(MODELS_DIR, 'race_lgbm_model.pkl'))
race_le      = joblib.load(os.path.join(MODELS_DIR, 'race_label_encoder.pkl'))

print('Gender model :', type(gender_model).__name__)
print('Race model   :', type(race_model).__name__)
print(f'Gender TF-IDF features: {len(gender_vec.get_feature_names_out()):,}')
print(f'Race   TF-IDF features: {len(race_vec.get_feature_names_out()):,}')
print('Race classes :', race_le.classes_)

## 2. Core Prediction Functions

In [ ]:
def predict_gender(first_name: str) -> dict:
    """Predict gender and return label + confidence."""
    feats   = gender_vec.transform([first_name.lower()])
    label   = gender_model.predict(feats)[0]
    probs   = gender_model.predict_proba(feats)[0]
    classes = gender_model.classes_
    conf    = float(probs[list(classes).index(label)])
    return {'gender': label, 'confidence': conf}


def predict_race(last_name: str) -> dict:
    """Predict race and return label + probabilities per class."""
    feats     = race_vec.transform([last_name.lower()])
    idx       = race_model.predict(feats)[0]
    label     = race_le.inverse_transform([idx])[0]
    probs     = race_model.predict_proba(feats)[0]
    prob_dict = dict(zip(race_le.classes_, probs.round(4)))
    return {'race': label, 'probabilities': prob_dict}


def predict_demographics(first_name: str, last_name: str) -> dict:
    """Combined gender + race prediction for a full name."""
    g = predict_gender(first_name)
    r = predict_race(last_name)
    return {
        'first_name':        first_name,
        'last_name':         last_name,
        'gender':            g['gender'],
        'gender_confidence': g['confidence'],
        'race':              r['race'],
        'race_probabilities': r['probabilities'],
    }

## 3. Single-Name Examples

In [ ]:
result = predict_demographics('Khondhaker', 'Momin')
print('Prediction for Khondhaker Momin')
print(f'  Gender : {result["gender"]}  (confidence: {result["gender_confidence"]:.3f})')
print(f'  Race   : {result["race"]}')
print('  Race probabilities:')
for cls, prob in sorted(result['race_probabilities'].items(), key=lambda x: -x[1]):
    bar = '█' * int(prob * 30)
    print(f'    {cls:<16} {prob:.4f}  {bar}')

## 4. Batch Prediction

In [ ]:
sample_data = pd.DataFrame({
    'first_name': ['Mary',    'James',   'Maria',     'David',   'Ashley',
                   'Carlos',  'Jennifer', 'Wei',      'Fatima',  'Jordan'],
    'last_name':  ['Smith',   'Johnson',  'Garcia',   'Williams','Brown',
                   'Rodriguez','Kim',     'Chen',     'Hassan',  'Taylor'],
})
sample_data

In [ ]:
def batch_predict(df: pd.DataFrame,
                  first_col: str = 'first_name',
                  last_col:  str = 'last_name') -> pd.DataFrame:
    """Apply gender and race predictions to every row."""
    first_names = df[first_col].values
    last_names  = df[last_col].values

    Xg = gender_vec.transform([n.lower() for n in first_names])
    Xr = race_vec.transform([n.lower() for n in last_names])

    gender_labels = gender_model.predict(Xg)
    gender_probs  = gender_model.predict_proba(Xg).max(axis=1).round(4)
    race_idx      = race_model.predict(Xr)
    race_labels   = race_le.inverse_transform(race_idx)

    out = df.copy()
    out['predicted_gender']  = gender_labels
    out['gender_confidence'] = gender_probs
    out['predicted_race']    = race_labels
    return out


results_df = batch_predict(sample_data)
results_df

## 5. Visualise Batch Results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

g_counts = results_df['predicted_gender'].value_counts()
axes[0].pie(g_counts, labels=g_counts.index, autopct='%1.0f%%',
            colors=['#DD8452', '#4C72B0'])
axes[0].set_title('Predicted Gender Distribution')

r_counts = results_df['predicted_race'].value_counts()
axes[1].bar(r_counts.index, r_counts.values, color='#55A868')
axes[1].set_title('Predicted Race Distribution')
axes[1].set_xlabel('Race')
axes[1].set_ylabel('Count')
plt.xticks(rotation=20)

plt.tight_layout()
plt.show()

## 6. Confidence Threshold Filtering

For downstream research, you may want to retain only high-confidence predictions.

In [ ]:
THRESHOLD = 0.70

high_conf = results_df[results_df['gender_confidence'] >= THRESHOLD]
print(f'Records above {THRESHOLD*100:.0f}% gender confidence: {len(high_conf)}/{len(results_df)}')
high_conf[['first_name', 'last_name', 'predicted_gender', 'gender_confidence', 'predicted_race']]

## 7. Probability Output — Race Detail

In [ ]:
# Show full race probability distribution for each sample
Xr = race_vec.transform([n.lower() for n in sample_data['last_name'].values])
probs = race_model.predict_proba(Xr)

print(f'{"Name":<22}', '  '.join(f'{c[:8]:>8}' for c in race_le.classes_))
print('-' * 80)
for i, (fn, ln) in enumerate(zip(sample_data['first_name'], sample_data['last_name'])):
    prob_str = '  '.join(f'{p:>8.3f}' for p in probs[i])
    print(f'{fn+" "+ln:<22} {prob_str}')